1. Датасет ml-latest.
2. Вспомнить подходы, которые мы разбирали.
3. Выбрать понравившийся подход к гибридным системам.
4. Написать свою.

In [58]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

import warnings
warnings.filterwarnings('ignore')

In [3]:
df_movies = pd.read_csv('ml_latest_small/movies.csv')
df_ratings = pd.read_csv('ml_latest_small/ratings.csv')

In [4]:
print(df_movies.shape)
print(df_ratings.shape)

(9742, 3)
(100836, 4)


In [5]:
df_movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [6]:
df_ratings.head()

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


## Контентная фильтрация:

In [7]:
df_movies.genres = df_movies.genres.str.split('|')
df_movies.head()

,movieId,title,genres
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]"
1,2,Jumanji (1995),"[Adventure, Children, Fantasy]"
2,3,Grumpier Old Men (1995),"[Comedy, Romance]"
3,4,Waiting to Exhale (1995),"[Comedy, Drama, Romance]"
4,5,Father of the Bride Part II (1995),[Comedy]


In [8]:
all_genres = set()

for genres in df_movies.genres:
    all_genres.update(genres)

all_genres = sorted(list(all_genres))

In [9]:
movie_genres_matrix = np.zeros(shape = (len(df_movies), len(all_genres)))
movie_genres_matrix

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(9742, 20))

In [10]:
df_movies['genres']

0       [Adventure, Animation, Children, Comedy, Fantasy]
1                          [Adventure, Children, Fantasy]
2                                       [Comedy, Romance]
3                                [Comedy, Drama, Romance]
4                                                [Comedy]
                              ...                        
9737                 [Action, Animation, Comedy, Fantasy]
9738                         [Animation, Comedy, Fantasy]
9739                                              [Drama]
9740                                  [Action, Animation]
9741                                             [Comedy]
Name: genres, Length: 9742, dtype: object

In [11]:
for i, genres in enumerate(df_movies['genres']):
    for genre in genres:
        genre_idx = all_genres.index(genre)
        movie_genres_matrix[i, genre_idx] = 1

movie_genres_matrix

array([[0., 0., 1., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 1., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(9742, 20))

In [12]:
movie_avg_ratings = df_ratings.groupby('movieId')['rating'].mean()
movie_avg_ratings

movieId
1         3.920930
2         3.431818
3         3.259615
4         2.357143
5         3.071429
            ...   
193581    4.000000
193583    3.500000
193585    3.500000
193587    3.500000
193609    4.000000
Name: rating, Length: 9724, dtype: float64

## Коллаборативная фильтрация:

In [66]:
user_id = 1
n_recommendations = 10

user_ratings = df_ratings[df_ratings['userId'] == user_id]
linked_movies = user_ratings[user_ratings['rating'] >= 4.0]['movieId'].values

if len(linked_movies) == 0:
    top_movies = movie_avg_ratings.sort_values(ascending = False).head(n_recommendations).index

pd.DataFrame({'movieId': top_movies, 'score': 1.0})

linked_indices = df_movies[df_movies['movieId'].isin(linked_movies)].index

user_genre_profile = movie_genres_matrix[linked_indices].mean(axis = 0)

similarities = cosine_similarity([user_genre_profile], movie_genres_matrix)[0]

array([0.64201704, 0.50595882, 0.38321726, ..., 0.36898795, 0.41990828,
       0.40358057], shape=(9742,))

In [65]:
print(user_genre_profile.shape)
print(movie_genres_matrix.shape)

(20,)
(9742, 20)
